# Week 9 · Implementing RSA in Python
The code of the handout, one section per cell. Run the cells in order.

## 0 · The reference game

In [1]:
from math import log2

objects = ["blue square", "green square",
           "blue circle", "green circle"]
prior = {"blue square": 0.44, "green square": 0.09,
         "blue circle": 0.28, "green circle": 0.19}
words = ["blue", "green", "square", "circle"]

def meaning(u, s):
    return all(w in s for w in u.split())

def show(d):
    for k, v in d.items():
        print(k, round(v, 2))

## 1 · The literal listener

In [2]:
def L0(u):
    total = sum(meaning(u, s) * prior[s] for s in objects)
    return {s: meaning(u, s) * prior[s] / total for s in objects}

show(L0("blue"))

blue square 0.61
green square 0.0
blue circle 0.39
green circle 0.0


## 2 · The speaker

In [3]:
def S1(s, alpha=1, utterances=None, cost=lambda u: 0):
    scores = {u: 2 ** (alpha * (log2(L0(u)[s]) - cost(u)))
              for u in (utterances or words) if meaning(u, s)}
    total = sum(scores.values())
    return {u: scores[u] / total for u in scores}

show(S1("blue square"))

blue 0.42
square 0.58


## 3 · The pragmatic listener

In [4]:
def L1(u, alpha=1):
    products = {s: S1(s, alpha).get(u, 0) * prior[s]
                for s in objects}
    evidence = sum(products.values())
    return {s: products[s] / evidence for s in objects}

show(L1("circle"))

blue square 0.0
green square 0.0
blue circle 0.7
green circle 0.3


## 4 · Costs

In [5]:
signals = words + ["blue square", "green square",
                   "blue circle", "green circle"]
cost = lambda u: len(u.split()) - 1

show(S1("blue square", alpha=2, utterances=signals, cost=cost))

blue 0.28
square 0.53
blue square 0.19


## 5 · Scalar implicature

In [6]:
objects = [0, 1, 2, 3]
prior = {0: 0.25, 1: 0.25, 2: 0.25, 3: 0.25}
words = ["none", "some", "all"]

def meaning(u, n):
    return {"none": n == 0, "some": n > 0, "all": n == 3}[u]

show(S1(3))
show(L1("some"))

some 0.25
all 0.75
0 0.0
1 0.44
2 0.44
3 0.11
